# FingerBall v2 Stage 2B — Manual anchor, bidirectional tracking, WiLoR 21-joint QC
Colab: https://colab.research.google.com/github/atstuyuki/FingerBall-Analysis/blob/main/notebooks/FingerBall_v2_Stage2B_Anchored_Tracking_21Landmarks.ipynb

21点の2D表示は未検証の暫定投影です。正規のWiLoR前処理・カメラ式が検証されるまで定量解析には使わないでください。


## 0. 環境セットアップ

In [ ]:
%pip -q install "openvino>=2025,<2027" "mediapipe==0.10.35" "opencv-python-headless>=4.8" matplotlib pandas pyyaml tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. 設定

In [ ]:
from pathlib import Path
import os,time,json,shutil,urllib.request
import numpy as np,pandas as pd,cv2,openvino as ov,mediapipe as mp,yaml
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from google.colab import files
PROJECT_NAME='母指回内トラッキング'
PROJECT_PATH=''
VIDEO_PATH=''
THROWING_HAND='right'
YOLO_CLASS_RIGHT=1
CONF_MIN=0.10
POSE_VIS_MIN=0.35
WRIST_GATE=1.4
FLOW_MAX_AGE=12
FLOW_MIN_VALID=0.45
STRIDE=1
RUN_WILOR=True
CACHE=Path('/content/fingerball_stage2a_cache')
RESULTS_NAME='FingerBall_v2_stage2b_results'
POSE_TASK_URL='https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker/float16/latest/pose_landmarker_full.task'
assert THROWING_HAND in ('right','left') and STRIDE>=1
print('OpenVINO:',ov.__version__,'MediaPipe:',mp.__version__)

## 2. Google Drive上の既存モデルをコピー

In [ ]:
def find_project():
    if PROJECT_PATH:
        p=Path(PROJECT_PATH)
        if not p.is_dir():raise FileNotFoundError(p)
        return p
    for root in [Path('/content/drive/MyDrive'),Path('/content/drive/Shareddrives')]:
        if (root/PROJECT_NAME).is_dir():return root/PROJECT_NAME
        if root.name=='Shareddrives' and root.is_dir():
            for d in root.iterdir():
                if (d/PROJECT_NAME).is_dir():return d/PROJECT_NAME
    raise FileNotFoundError('PROJECT_PATHを指定してください')
project=find_project()
a=project/'thumb_pronation/out/bench_light/verify/int8_leftcal'
b=project/'models/_DATA/wilor/pretrained_models/detector_openvino_model_288x512'
sources=[a/'wilor_core_int8_leftcal.xml',a/'wilor_core_int8_leftcal.bin',b/'detector.xml',b/'detector.bin',b/'metadata.yaml']
if not all(p.is_file() for p in sources):raise FileNotFoundError([(str(p),p.is_file()) for p in sources])
CACHE.mkdir(exist_ok=True,parents=True)
for src in sources:
    folder=CACHE/('wilor' if 'wilor_core' in src.name else 'detector')
    folder.mkdir(exist_ok=True)
    dest=folder/src.name
    if not dest.exists() or dest.stat().st_size!=src.stat().st_size:shutil.copy2(src,dest)
meta=yaml.safe_load((CACHE/'detector/metadata.yaml').read_text(encoding='utf8'))
print('Source:',project,'Hand classes:',meta.get('names'))

## 3. MediaPipe Pose Task / OpenVINOを準備

In [ ]:
task = CACHE / 'pose_landmarker_full.task'
POSE_TASK_URLS = [
    'https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/latest/pose_landmarker_full.task',
    'https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/1/pose_landmarker_full.task',
]
# 壊れた/途中までのファイルは再取得
if task.exists() and task.stat().st_size < 1000000:
    task.unlink()
if not task.is_file():
    errors = []
    for url in POSE_TASK_URLS:
        tmp = task.with_suffix('.download')
        try:
            print('MediaPipe Poseモデル取得:', url)
            request = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(request, timeout=90) as response, open(tmp, 'wb') as out:
                shutil.copyfileobj(response, out)
            if tmp.stat().st_size < 1000000:
                raise ValueError(f'不完全なモデル: {tmp.stat().st_size} bytes')
            tmp.replace(task)
            print('取得成功:', round(task.stat().st_size / 1e6, 1), 'MB')
            break
        except Exception as e:
            errors.append(f'{url}: {type(e).__name__}: {e}')
            tmp.unlink(missing_ok=True)
    if not task.is_file():
        raise RuntimeError('Poseモデルを取得できません。\\n' + '\\n'.join(errors))
assert task.stat().st_size > 1000000, 'Poseモデルのファイルサイズが不正です'
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
po=mp_vision.PoseLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=str(task)),
    running_mode=mp_vision.RunningMode.VIDEO,num_poses=1,
    min_pose_detection_confidence=.4,min_pose_presence_confidence=.4,min_tracking_confidence=.4)
pose=mp_vision.PoseLandmarker.create_from_options(po)
core=ov.Core()
det=core.compile_model(core.read_model(str(CACHE/'detector/detector.xml')),'CPU')
wilor=core.compile_model(core.read_model(str(CACHE/'wilor/wilor_core_int8_leftcal.xml')),'CPU')
dims=list(det.input(0).shape)
assert len(dims)==4 and dims[1]==3,dims
net_h,net_w=map(int,dims[2:])
nc=len(meta.get('names',{0:'left',1:'right'}))
print('Detector:',dims,'WiLoR inputs:',[list(p.shape) for p in wilor.inputs])
print('WiLoR outputs:',[list(p.shape) for p in wilor.outputs])

## 4. 動画を読み込む

In [ ]:
if not VIDEO_PATH:
    uploaded=files.upload()
    if not uploaded:raise RuntimeError('動画未選択')
    VIDEO_PATH=str(Path('/content')/next(iter(uploaded)))
video_src=Path(VIDEO_PATH)
if not video_src.is_file():raise FileNotFoundError(video_src)
source=video_src
if str(video_src).startswith('/content/drive/'):
    local=CACHE/('input_'+video_src.name)
    if not local.exists() or local.stat().st_size!=video_src.stat().st_size:shutil.copy2(video_src,local)
    source=local
cap=cv2.VideoCapture(str(source))
if not cap.isOpened():raise RuntimeError('OpenCVで動画を開けません')
total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT));fps=float(cap.get(cv2.CAP_PROP_FPS))
W=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH));H=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
ok,im=cap.read();cap.release()
if not ok:raise RuntimeError('動画が読めません')
print('Movie:',video_src.name,'frames:',total,'fps:',fps,'image:',W,H)
plt.figure(figsize=(10,5));plt.imshow(cv2.cvtColor(im,cv2.COLOR_BGR2RGB));plt.axis('off');plt.show()

## 5. 検出・腕ROI・LK光学フローの関数

出力の source は yolo_pose / yolo_flow / flow / pose_fallback / none。YOLOとROIの一致が得られた場合だけ yolo_validated=1 とします。

In [ ]:
def clip_box(b):
    if b is None:return None
    a=np.asarray(b,float).copy()
    if a.shape!=(4,) or not np.isfinite(a).all():return None
    a[[0,2]]=np.clip(a[[0,2]],0,W-1);a[[1,3]]=np.clip(a[[1,3]],0,H-1)
    return a if a[2]-a[0]>=8 and a[3]-a[1]>=8 else None
def ctr(b):return (b[:2]+b[2:])/2
def boxside(b):return max(b[2]-b[0],b[3]-b[1])
def rect(x,y,s):return clip_box([x-s/2,y-s/2,x+s/2,y+s/2])
def nms(boxes,scores,thr=.7):
    if not len(boxes):return []
    boxes=np.asarray(boxes);idx=np.argsort(-np.asarray(scores));keep=[]
    while len(idx):
        i=int(idx[0]);keep.append(i);rest=idx[1:]
        if not len(rest):break
        mn=np.maximum(boxes[i,:2],boxes[rest,:2]);mx=np.minimum(boxes[i,2:],boxes[rest,2:])
        intersection=np.prod(np.maximum(mx-mn,0),axis=1)
        a=np.prod(np.maximum(boxes[i,2:]-boxes[i,:2],0))
        b=np.prod(np.maximum(boxes[rest,2:]-boxes[rest,:2],0),axis=1)
        idx=rest[intersection/(a+b-intersection+1e-8)<=thr]
    return keep
def letterbox(img):
    h,w=img.shape[:2];scale=min(net_h/h,net_w/w)
    nw,nh=int(round(w*scale)),int(round(h*scale))
    dx,dy=(net_w-nw)/2,(net_h-nh)/2
    l,rr=int(round(dx-.1)),int(round(dx+.1))
    t,bb=int(round(dy-.1)),int(round(dy+.1))
    z=cv2.resize(img,(nw,nh),interpolation=cv2.INTER_LINEAR)
    z=cv2.copyMakeBorder(z,t,bb,l,rr,cv2.BORDER_CONSTANT,value=(114,114,114))
    return np.ascontiguousarray(z[None,...,::-1].transpose(0,3,1,2)).astype(np.float32)/255.,scale,l,t
def detect_hands(img):
    x,scale,l,t=letterbox(img)
    raw=np.asarray(det([x])[det.output(0)])
    if raw.ndim!=3 or raw.shape[0]!=1:raise RuntimeError('YOLO tensor shape: '+str(raw.shape))
    p=raw[0]
    if p.shape[0]>=4+nc:p=p.T
    if p.shape[1]<4+nc:raise RuntimeError('YOLO channels: '+str(p.shape))
    scores=p[:,4:4+nc]
    target=YOLO_CLASS_RIGHT if THROWING_HAND=='right' else 1-YOLO_CLASS_RIGHT
    conf=scores[:,target]
    k=conf>=CONF_MIN
    z=p[k,:4];cc=conf[k]
    if not len(z):return [],float(conf.max())
    boxes=np.column_stack([z[:,0]-z[:,2]/2,z[:,1]-z[:,3]/2,z[:,0]+z[:,2]/2,z[:,1]+z[:,3]/2])
    out=[]
    for i in nms(boxes,cc):
        b=boxes[i].copy()
        b[[0,2]]=(b[[0,2]]-l)/scale;b[[1,3]]=(b[[1,3]]-t)/scale
        b=clip_box(b)
        if b is not None:out.append((b,float(cc[i])))
    return out,float(conf.max())
def pose_arm(img,t_ms):
    image=mp.Image(image_format=mp.ImageFormat.SRGB,data=cv2.cvtColor(img,cv2.COLOR_BGR2RGB))
    result=pose.detect_for_video(image,int(t_ms))
    if not result.pose_landmarks:return None
    landmarks=result.pose_landmarks[0]
    ids=(12,14,16) if THROWING_HAND=='right' else (11,13,15)
    v=[float(landmarks[i].visibility) for i in ids]
    if min(v)<POSE_VIS_MIN:return None
    sh,el,wr=[np.array([landmarks[i].x*W,landmarks[i].y*H]) for i in ids]
    fore=float(np.linalg.norm(el-wr))
    if not np.isfinite(fore) or fore<12:return None
    return {'shoulder':sh,'elbow':el,'wrist':wr,'forearm':fore,'visibility':min(v)}
def flow_box(prev_gray,gray,b):
    if prev_gray is None or b is None:return None,0.
    bb=np.rint(b).astype(int)
    x0,y0,x1,y1=bb
    mask=np.zeros((H,W),np.uint8);mask[max(0,y0):min(H,y1),max(0,x0):min(W,x1)]=255
    p0=cv2.goodFeaturesToTrack(prev_gray,maxCorners=80,qualityLevel=.01,minDistance=4,mask=mask)
    if p0 is None or len(p0)<6:return None,0.
    p1,s1,_=cv2.calcOpticalFlowPyrLK(prev_gray,gray,p0,None,winSize=(21,21),maxLevel=3)
    if p1 is None:return None,0.
    p2,s2,_=cv2.calcOpticalFlowPyrLK(gray,prev_gray,p1,None,winSize=(21,21),maxLevel=3)
    if p2 is None:return None,0.
    fb=np.linalg.norm((p2-p0).reshape(-1,2),axis=1)
    valid=(s1[:,0]>0)&(s2[:,0]>0)&(fb<1.5)
    frac=float(valid.mean())
    if valid.sum()<6 or frac<FLOW_MIN_VALID:return None,frac
    shift=np.median((p1-p0).reshape(-1,2)[valid],axis=0)
    return clip_box(b+np.tile(shift,2)),frac
def patch_for_wilor(img,box):
    # Stage2A近似切り出し: オリジナルのガウシアン前処理との一致は未検証
    cx,cy=ctr(box);s=2.5*boxside(box)
    if THROWING_HAND=='left':
        img=cv2.flip(img,1);cx=W-1-cx
    M=np.array([[256/s,0,128-cx*256/s],[0,256/s,128-cy*256/s]],np.float32)
    return cv2.warpAffine(img,M,(256,256),flags=cv2.INTER_LINEAR).astype(np.float32)


## 6. 投球手の手動初期化（Mac / Colab対応・スライダー方式）

1. `MANUAL_FRAME` に投球手が見えるフレーム番号を指定し、このセルを実行。
2. X座標・Y座標の範囲スライダーを調整し、**緑色のBOXで投球手を囲む**。
3. **「このBOXで確定」ボタンを押す**。
4. 次の「確定BOX確認」セルを実行し、以降の追跡セルへ進む。

トラックパッドでの直接ドラッグ選択は使用しません。スライダーに数値を入力・操作できるほか、`MANUAL_BOX=(x0,y0,x1,y1)` の直接指定も可能です。**「すべて実行」は使わず、ここで一旦操作を完了してください。**

In [ ]:
MANUAL_FRAME = 180
MANUAL_BOX = None  # 直接指定する場合: (x0, y0, x1, y1)
RELEASE_START, RELEASE_END = 175, 210
assert 0 <= MANUAL_FRAME < total
ANCHOR_CONFIRMED = False
anchor_box = None
cc = cv2.VideoCapture(str(source))
cc.set(cv2.CAP_PROP_POS_FRAMES, MANUAL_FRAME)
ok, anchor_img = cc.read()
cc.release()
if not ok:
    raise RuntimeError(f"Frame {MANUAL_FRAME} could not be decoded")

import ipywidgets as widgets
from IPython.display import display
from google.colab import output
output.enable_custom_widget_manager()

print(f"元画像サイズ: {W} x {H}, 選択フレーム: {MANUAL_FRAME}")
print("X/Y範囲を調整し、緑枠が投球手を囲んだら「このBOXで確定」を押してください。")

def roi_preview(box):
    vis = anchor_img.copy()
    if box is not None:
        x0, y0, x1, y1 = np.rint(box).astype(int)
        cv2.rectangle(vis, (x0,y0), (x1,y1), (0,255,0), 3)
        cv2.putText(vis, f"ROI: {x0},{y0},{x1},{y1}", (12,32),
                    cv2.FONT_HERSHEY_SIMPLEX, .8, (0,255,0), 2)
    scale = min(1.0, 960/max(W,1))
    if scale < 1:
        vis = cv2.resize(vis, None, fx=scale, fy=scale)
    ok, buf = cv2.imencode(".png", vis)
    return buf.tobytes()

x_range = widgets.IntRangeSlider(value=(int(W*.40),int(W*.60)), min=0, max=W-1,
    step=1, description="X範囲", continuous_update=False, layout=widgets.Layout(width="95%"))
y_range = widgets.IntRangeSlider(value=(int(H*.30),int(H*.60)), min=0, max=H-1,
    step=1, description="Y範囲", continuous_update=False, layout=widgets.Layout(width="95%"))
image_widget = widgets.Image(value=roi_preview(None), format="png")
confirm_btn = widgets.Button(description="このBOXで確定", button_style="success", icon="check")
status = widgets.HTML(value="<b>未確定</b>：緑枠を手に合わせてください")
def current_box():
    return clip_box((x_range.value[0], y_range.value[0], x_range.value[1], y_range.value[1]))
def on_range_change(change):
    global ANCHOR_CONFIRMED
    ANCHOR_CONFIRMED = False
    image_widget.value = roi_preview(current_box())
    status.value = "<b>未確定</b>：変更後はもう一度「このBOXで確定」を押してください"
x_range.observe(on_range_change, names="value")
y_range.observe(on_range_change, names="value")
def on_confirm(button):
    global anchor_box, MANUAL_BOX, ANCHOR_CONFIRMED
    b = current_box()
    if b is None:
        status.value = "<b>無効なBOX</b>：幅と高さを8px以上にしてください"
        return
    anchor_box = b.copy()
    MANUAL_BOX = tuple(map(float, b))
    ANCHOR_CONFIRMED = True
    status.value = "<b>確定しました：</b> " + str(tuple(map(int, np.rint(anchor_box))))
confirm_btn.on_click(on_confirm)
if MANUAL_BOX is not None:
    anchor_box = clip_box(MANUAL_BOX)
    if anchor_box is None: raise ValueError("MANUAL_BOX is invalid")
    ANCHOR_CONFIRMED = True
    status.value = "<b>直接指定のBOXが確定しました</b>"
    image_widget.value = roi_preview(anchor_box)
else:
    image_widget.value = roi_preview(current_box())
display(image_widget, x_range, y_range, confirm_btn, status)

In [ ]:
if not globals().get("ANCHOR_CONFIRMED", False) or anchor_box is None:
    raise RuntimeError("手のBOXが未確定です。直前のセルでスライダーを調整し「このBOXで確定」をクリックしてください。")
print("投球手アンカー確定:", MANUAL_FRAME, tuple(np.rint(anchor_box).astype(int)))
plt.figure(figsize=(12,6))
check = anchor_img.copy()
a=np.rint(anchor_box).astype(int)
cv2.rectangle(check, tuple(a[:2]), tuple(a[2:]), (0,255,0), 3)
plt.imshow(cv2.cvtColor(check,cv2.COLOR_BGR2RGB));plt.axis("off");plt.show()

## 7. Pose・YOLO候補の前計算

MediaPipe VIDEOモードは時刻順に実行し、その後に手動アンカーから双方向追跡します。

In [ ]:
pose_cache={};yolo_cache={}
cap=cv2.VideoCapture(str(source))
for i in tqdm(range(total),desc='Pose + YOLO precompute'):
    ok,img=cap.read()
    if not ok:break
    arm=pose_arm(img,int(round(i*1000/fps)) if fps>0 else i*33)
    pose_cache[i]=arm
    yolo_cache[i]=detect_hands(img)
cap.release();pose.close()
if len(pose_cache)!=total:raise RuntimeError('Some frames could not be decoded')
print('Pose valid:',sum(x is not None for x in pose_cache.values()))
print('YOLO candidates:',sum(bool(v[0]) for v in yolo_cache.values()))

## 8. アンカーを起点とした前後両方向の追跡

検出と光学フローの品質を別々に記録します。投球手を見失った後、別の位置のYOLO候補へ飛び移らないようにします。

In [ ]:
TRACK_GATE=1.45
def read_at(cap,i):
    cap.set(cv2.CAP_PROP_POS_FRAMES,int(i))
    ok,img=cap.read()
    if not ok:raise RuntimeError('Frame read failed '+str(i))
    return img
def run_direction(indices):
    cap=cv2.VideoCapture(str(source))
    prev_gray=None;box=anchor_box.copy();age=0;out={}
    for i in tqdm(indices,desc='Tracking'):
        img=read_at(cap,i)
        gray=cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
        arm=pose_cache[i]
        candidates,maxconf=yolo_cache[i]
        if i==MANUAL_FRAME:
            out[i]={'box':anchor_box.copy(),'source':'manual','pose_valid':int(arm is not None),
                    'score':np.nan,'flow_fraction':np.nan,'max_conf':maxconf,'age':0}
            prev_gray=gray
            continue
        predicted,flow_fraction=flow_box(prev_gray,gray,box)
        ref=predicted if predicted is not None else (box if age<=FLOW_MAX_AGE else None)
        found=None;score=np.nan;origin='none'
        if ref is not None:
            pool=[]
            for b,conf in candidates:
                dist=np.linalg.norm(ctr(b)-ctr(ref))/max(boxside(ref),15)
                if dist>TRACK_GATE:continue
                if arm is not None:
                    wrist_dist=np.linalg.norm(ctr(b)-arm['wrist'])/max(arm['forearm'],1)
                    if wrist_dist>WRIST_GATE:continue
                else:wrist_dist=0.
                pool.append((conf-.25*dist-.15*wrist_dist,b,conf))
            if pool:
                _,found,score=max(pool,key=lambda x:x[0])
                origin='yolo_pose' if arm is not None else 'yolo_track'
        if found is not None:
            box=found.copy();age=0
        elif predicted is not None and age<FLOW_MAX_AGE:
            box=predicted.copy();age+=1;origin='flow'
        else:
            box=None;age+=1;origin='none'
        out[i]={'box':box.copy() if box is not None else None,'source':origin,
                'pose_valid':int(arm is not None),'score':float(score),
                'flow_fraction':float(flow_fraction),'max_conf':maxconf,'age':age}
        prev_gray=gray
    cap.release()
    return out
forward=run_direction(list(range(MANUAL_FRAME,total)))
backward=run_direction(list(range(MANUAL_FRAME,-1,-1)))
tracking={**backward,**forward}
assert len(tracking)==total
print(pd.Series([tracking[i]['source'] for i in range(total)]).value_counts())

## 9. WiLoR推論・21点の暫定2D投影・フレーム別品質CSV

WiLoRの出力4＝21×3関節、出力3＝カメラ3値を想定する診断用投影です。**純正のカメラデコードや入力正規化と一致するか未検証**なので、この重畳が指を正しく示すか必ず目視確認してください。`wilor_finite` は精度指標ではありません。

In [ ]:
RESULTS_NAME='FingerBall_v2_stage2b_results'
out_dir=project/RESULTS_NAME/video_src.stem
out_dir.mkdir(parents=True,exist_ok=True)
SKELETON=[(0,1),(1,2),(2,3),(3,4),(0,5),(5,6),(6,7),(7,8),
          (0,9),(9,10),(10,11),(11,12),(0,13),(13,14),(14,15),(15,16),
          (0,17),(17,18),(18,19),(19,20)]
def project_weak_perspective(j3d,cam,box):
    joints=np.asarray(j3d).reshape(21,3)
    cp=np.asarray(cam).reshape(-1)
    if cp.size<3:return None
    # Diagnostic approximation ONLY: UV_patch=128*(scale*J_xy+trans+1)
    uv=128.0*(cp[0]*joints[:,:2]+cp[None,1:3]+1.0)
    c=ctr(box);s=2.5*boxside(box)
    xy=c[None,:]+(uv-128.)*s/256.
    if THROWING_HAND=='left':xy[:,0]=2*c[0]-xy[:,0]
    return xy if np.isfinite(xy).all() else None
cap=cv2.VideoCapture(str(source))
rows=[];raw_outputs=[];landmarks={}
for i in tqdm(range(total),desc='WiLoR inference'):
    img=read_at(cap,i)
    state=tracking[i];box=state['box']
    row={'frame':i,'time_s':i/fps if fps>0 else np.nan,'source':state['source'],
         'has_roi':int(box is not None),'yolo_confirmed':int(state['source'].startswith('yolo')),
         'flow_only':int(state['source']=='flow'),'manual_anchor':int(state['source']=='manual'),
         'no_roi':int(box is None),'pose_valid':state['pose_valid'],
         'wilor_finite':0,'landmarks_provisional':0,'selected_yolo_conf':state['score'],
         'max_yolo_conf':state['max_conf'],'flow_valid_fraction':state['flow_fraction'],
         'frames_since_yolo':state['age']}
    if box is not None:
        row.update({k:float(v) for k,v in zip(('x0','y0','x1','y1'),box)})
        if RUN_WILOR:
            try:
                patch=patch_for_wilor(img,box)
                input_shape=list(wilor.input(0).shape)
                x=patch[None] if input_shape[-1]==3 else patch.transpose(2,0,1)[None]
                ret=wilor([np.ascontiguousarray(x)])
                vals=[np.asarray(ret[p]).copy() for p in wilor.outputs]
                if all(np.isfinite(v).all() for v in vals):
                    row['wilor_finite']=1
                    raw_outputs.append((i,vals))
                    if len(vals)>4 and vals[4].size==63 and vals[3].size>=3:
                        xy=project_weak_perspective(vals[4],vals[3],box)
                        if xy is not None:
                            landmarks[i]=xy
                            row['landmarks_provisional']=1
            except Exception as exc:row['wilor_error']=str(exc)
    rows.append(row)
cap.release()
df=pd.DataFrame(rows)
qc_csv=out_dir/'tracking_qc.csv'
df.to_csv(qc_csv,index=False)
raw_npz=out_dir/'wilor_raw_outputs.npz'
if raw_outputs:
    payload={'frame_idx':np.asarray([i for i,_ in raw_outputs])}
    for k in range(len(raw_outputs[0][1])):
        payload['output_'+str(k)]=np.concatenate([o[k] for _,o in raw_outputs],axis=0)
    np.savez_compressed(raw_npz,**payload)
landmark_npz=out_dir/'landmarks_2d_PROVISIONAL.npz'
if landmarks:
    keys=sorted(landmarks)
    np.savez_compressed(landmark_npz,frame_idx=np.asarray(keys),
                        xy=np.stack([landmarks[k] for k in keys]))
print(df.source.value_counts().to_string())
print('WiLoR finite:',int(df.wilor_finite.sum()),'Provisional landmark frames:',len(landmarks))
print('Saved:',qc_csv)

## 10. 全フレーム重畳動画と175〜210拡大動画

緑：YOLO一致、黄色：光学フロー、青：手動アンカー、赤：追跡失敗。21点は**暫定投影の白線・赤点**です。拡大動画のクロップ範囲はアンカーBOXと追跡BOXの移動に合わせ、見失ったときは直近ROIを使用します。

In [ ]:
COLORS={'manual':(255,100,0),'yolo_pose':(0,220,0),'yolo_track':(0,190,80),
        'flow':(0,255,255),'none':(0,0,255)}
def draw_overlay(img,i):
    row=df.iloc[i]
    color=COLORS.get(row['source'],(120,120,120))
    if int(row['has_roi']):
        b=np.rint([row[k] for k in ('x0','y0','x1','y1')]).astype(int)
        cv2.rectangle(img,tuple(b[:2]),tuple(b[2:]),color,2)
    if i in landmarks:
        xy=landmarks[i]
        # Out-of-image landmarks are not silently clipped; QC can reveal projection errors.
        if np.max(np.abs(xy))<100000:
            pts=np.rint(xy).astype(int)
            for a,b in SKELETON:
                cv2.line(img,tuple(pts[a]),tuple(pts[b]),(255,255,255),2)
            for pt in pts:cv2.circle(img,tuple(pt),3,(20,20,255),-1)
    label=f"f={i} {row['source']} WiLoR={int(row['wilor_finite'])} 2D=PROVISIONAL"
    cv2.putText(img,label,(10,26),cv2.FONT_HERSHEY_SIMPLEX,.58,color,2)
    return img
full_video=out_dir/'stage2b_tracking_21points_PROVISIONAL.mp4'
cap=cv2.VideoCapture(str(source))
writer=cv2.VideoWriter(str(full_video),cv2.VideoWriter_fourcc(*'mp4v'),fps if fps>0 else 30,(W,H))
if not writer.isOpened():raise RuntimeError('Full video writer failed')
j=0
try:
    while True:
        ok,img=cap.read()
        if not ok:break
        writer.write(draw_overlay(img,j));j+=1
finally:
    cap.release();writer.release()
print('Full overlay:',full_video,'frames:',j)
# zoom video always covers requested frame range and uses a smooth previous ROI fallback
zoom_video=out_dir/f'release_zoom_{RELEASE_START}_{RELEASE_END}.mp4'
out_size=(768,768)
writer=cv2.VideoWriter(str(zoom_video),cv2.VideoWriter_fourcc(*'mp4v'),fps if fps>0 else 30,out_size)
if not writer.isOpened():raise RuntimeError('Zoom writer failed')
cap=cv2.VideoCapture(str(source))
last_center=ctr(anchor_box)
zoom_side=min(max(boxside(anchor_box)*5.0,180),max(W,H))
start=max(0,int(RELEASE_START));end=min(total-1,int(RELEASE_END))
try:
    for i in range(start,end+1):
        img=read_at(cap,i)
        bb=tracking[i]['box']
        if bb is not None:
            last_center=ctr(bb)
            zoom_side=min(max(boxside(bb)*5.0,180),max(W,H))
        view=draw_overlay(img,i)
        ss=max(64,int(round(zoom_side)))
        cx,cy=last_center
        M=np.float32([[out_size[0]/ss,0,out_size[0]/2-cx*out_size[0]/ss],
                      [0,out_size[1]/ss,out_size[1]/2-cy*out_size[1]/ss]])
        crop=cv2.warpAffine(view,M,out_size,borderMode=cv2.BORDER_CONSTANT,
                            borderValue=(30,30,30))
        cv2.putText(crop,f'Release QC frame {i}',(12,742),
                    cv2.FONT_HERSHEY_SIMPLEX,.8,(255,255,255),2)
        writer.write(crop)
finally:
    cap.release();writer.release()
print('Zoom:',zoom_video,'frames:',end-start+1)

## 11. QCサマリーと比較レポート

`yolo_confirmed`・`flow_only`・`wilor_finite`は異なる品質項目です。手の正解ラベルを付けるまでは、解析精度やイベント検出精度は未評価です。

In [ ]:
from datetime import datetime
report={'video':video_src.name,'created_at':datetime.now().isoformat(timespec='seconds'),
        'frames_total':total,'fps':fps,'manual_anchor_frame':MANUAL_FRAME,
        'manual_box_xyxy':[float(x) for x in anchor_box],
        'yolo_confirmed_frames':int(df.yolo_confirmed.sum()),
        'flow_only_frames':int(df.flow_only.sum()),
        'no_roi_frames':int(df.no_roi.sum()),
        'wilor_finite_frames':int(df.wilor_finite.sum()),
        'landmarks_provisional_frames':int(df.landmarks_provisional.sum()),
        'release_window':[start,end],
        'source_counts':{str(k):int(v) for k,v in df.source.value_counts().items()},
        'files':{'qc_csv':str(qc_csv),'raw_npz':str(raw_npz) if raw_npz.exists() else None,
                 'landmarks_npz':str(landmark_npz) if landmark_npz.exists() else None,
                 'full_overlay':str(full_video),'release_zoom':str(zoom_video)},
        'limitations':['Manual ROI is a single anchor, not all-frame ground truth',
                       'Optical flow ROI may drift during occlusion',
                       'WiLoR crop and input normalization are not validated against original',
                       'Weak-perspective 2D overlay is PROVISIONAL, not calibrated',
                       'No ball tracking or release estimation']}
json_path=out_dir/'stage2b_report.json'
json_path.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf8')
print(json.dumps(report,ensure_ascii=False,indent=2))
print('Results:',out_dir)
print('Check release overlay before kinematic interpretation')

## 重要：次の検証

重畳動画で**175〜210**の手指21点と投球手BOXを目視評価してください。前処理・カメラ投影が元のWiLoRと一致するまで2D座標を定量化しないでください。精度検証では、少なくともリリース付近の手ROIと手指21点の正解データを作ることを推奨します。